# Iowa Corn Monthly 10-Batch Extraction on Google Colab

This notebook runs the full Iowa Corn monthly CropNet extraction in Google Drive storage. It does not use or depend on any local Windows process.

Run order:

1. Mount Google Drive.
2. Verify the uploaded repo and USDA files.
3. Check current batch status.
4. Run missing batches one at a time by default.
5. Merge after all 10 batches validate.
6. Retrain the monthly yield model.
7. Print the final readiness report.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Define Drive Paths

Upload the project to `/content/drive/MyDrive/cropnet_iowa_corn/Crop-Net-repo` before running the rest of the notebook.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive/cropnet_iowa_corn')
REPO_ROOT = DRIVE_ROOT / 'Crop-Net-repo'
EXPERIMENT_ROOT = REPO_ROOT / 'outputs' / 'experiments'
PROJECT_DIR = EXPERIMENT_ROOT / 'corn_ia_monthly_2017_2022'
HF_CACHE_DIR = DRIVE_ROOT / 'hf_cache'

os.chdir(REPO_ROOT)
for path in [REPO_ROOT / 'src', REPO_ROOT / 'Crop-Net' / 'src']:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT / 'Crop-Net' / 'src'), os.environ.get('PYTHONPATH', '')])
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT =', DRIVE_ROOT)
print('REPO_ROOT =', REPO_ROOT)
print('EXPERIMENT_ROOT =', EXPERIMENT_ROOT)
print('HF_CACHE_DIR =', HF_CACHE_DIR)


## 3. Install Dependencies

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q h5py seaborn joblib huggingface_hub pyarrow


## 4. Verify Python, Packages, Imports, and Uploaded Files

In [ ]:
import importlib
import importlib.metadata as metadata
import importlib.util
import platform

print('Python:', sys.version)
print('Platform:', platform.platform())
for package in ['pandas', 'numpy', 'scikit-learn', 'joblib', 'matplotlib', 'pyarrow', 'h5py', 'seaborn', 'huggingface_hub']:
    try:
        print(f'{package}:', metadata.version(package))
    except metadata.PackageNotFoundError:
        print(f'{package}: NOT INSTALLED')

def import_file(module_name, path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

import cropnet_forecasting.yield_batching as yield_batching
import cropnet_forecasting.yield_regression as yield_regression
feature_server = import_file('cropnet_feature_forecasting_v12_server', REPO_ROOT / 'scripts' / 'research' / 'cropnet_feature_forecasting_v12_server.py')
print('Import verified: src/cropnet_forecasting/yield_batching.py')
print('Import verified: scripts/research/cropnet_feature_forecasting_v12_server.py')
print('Import verified: src/cropnet_forecasting/yield_regression.py')

required_paths = [
    REPO_ROOT / 'src',
    REPO_ROOT / 'scripts',
    REPO_ROOT / 'configs',
    REPO_ROOT / 'Crop-Net' / 'src',
    REPO_ROOT / 'requirements.txt',
    REPO_ROOT / 'pyproject.toml',
    PROJECT_DIR / 'batch_manifest.csv',
    PROJECT_DIR / 'batch_commands.txt',
]
required_paths += [
    REPO_ROOT / 'data' / 'usda_labels' / 'USDA Crop Dataset' / 'Corn' / str(year) / f'USDA_Corn_County_{year}.csv'
    for year in range(2017, 2023)
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    print('Missing required uploads:')
    for path in missing:
        print(' -', path)
    raise FileNotFoundError('Upload the missing files to Google Drive before continuing.')
print('All required uploads are present.')


## 5. Helper Function for Runner Commands

In [ ]:
HELPER = REPO_ROOT / 'scripts' / 'colab_run_iowa_corn_10batch.py'

def run_helper(args, check=False):
    cmd = [
        sys.executable,
        str(HELPER),
        '--repo-root', str(REPO_ROOT),
        '--hf-cache-dir', str(HF_CACHE_DIR),
        *args,
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')
    return result

subprocess.run([sys.executable, '-m', 'py_compile', str(HELPER)], cwd=REPO_ROOT, check=True)
print('Helper script compiles:', HELPER)


## 6. Show Completed, Missing, and Invalid Batches

This cell does not run extraction. It validates existing batch artifacts and writes `colab_batch_status.json` to Drive.

In [ ]:
run_helper(['status'], check=False)


## 7. Run One Missing Batch

Use this when you want controlled progress. Set `BATCH_ID` to a number from 1 to 10. If that batch already has a valid final parquet artifact, it will be skipped.

In [ ]:
BATCH_ID = 1
run_helper(['run-one', '--batch-id', str(BATCH_ID)], check=False)
run_helper(['status'], check=False)


## 8. Resume From Last Incomplete Batch

This is the normal Colab resume cell. It skips valid artifacts and runs missing or invalid batches sequentially. Re-run this cell after a disconnect.

In [ ]:
run_helper(['run-missing', '--parallelism', '1'], check=False)
run_helper(['status'], check=False)


## 9. Optional Experimental Parallelism

Keep this disabled unless you have a high-RAM Colab runtime and enough Drive/network headroom. Sequential execution is safer.

In [ ]:
RUN_PARALLEL_2 = False
if RUN_PARALLEL_2:
    run_helper(['run-missing', '--parallelism', '2'], check=False)
    run_helper(['status'], check=False)
else:
    print('Parallelism disabled. Set RUN_PARALLEL_2 = True only if the runtime can handle it.')


## 10. Merge After All 10 Batches Validate

This cell refuses to merge unless all 10 batch artifacts are valid.

In [ ]:
run_helper(['merge'], check=True)


## 11. Retrain Monthly Yield Model

This trains on the merged real monthly feature table and USDA annual labels copied to monthly rows.

In [ ]:
run_helper(['retrain'], check=True)


## 12. Final Report

The report includes completed batch count, completed/missing/invalid batch numbers, merged path, retrain output path, model path, metrics, and readiness for yield prediction.

In [ ]:
run_helper(['final-report'], check=False)
